# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR<sup>2</sup> dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library, following the Croissant schema standard.

### Dataset Source
The dataset is described using a Croissant schema accessible at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}\n")

## 2. Data Overview
Display available record sets, their `@id`s, fields, and structure to understand the dataset contents.

Below we inspect the record sets, fields, columns, and their unique `@id`s using the Croissant metadata API.

In [ ]:
# List all record sets and their fields, referencing everything by '@id'
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s) in the dataset.\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name}: @id={field.id}, type={field.data_type}")
    print()
# Select the first record set for further analysis
chosen_record_set = record_sets[0] if record_sets else None
# Collect all numeric fields for later use
numeric_fields = [f for f in chosen_record_set.fields if f.data_type in ['Float', 'Integer', 'Number']]
print("Numeric fields in the record set and their @id:")
for f in numeric_fields:
    print(f" - {f.name}: @id={f.id}")

## 3. Data Extraction
Load all available record sets into pandas DataFrames.
We use the record set and field `@id`s from the previous overview and reference everything by its `@id`.

In [ ]:
# Extract data from all record sets and store in a dictionary (keyed by @id).
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    print(f"Loading records for record set: {rs.name} (id: {rs.id}) ...")
    records = list(dataset.records(record_set=rs.id))
    if len(records) == 0:
        print("  No records found.")
    dataframes[rs.id] = pd.DataFrame(records)
    print(f"  Shape: {dataframes[rs.id].shape}")
    print()

# Preview the first record set DataFrame and its columns
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in record set '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filter and normalize a numeric field, and group data by a relevant field.

Be sure to reference fields and columns via their `@id`.

Below we demonstrate:
- Filtering rows based on a threshold in a numeric field
- Normalizing that field
- Grouping by a categorical field

In [ ]:
# Choose a numeric field and a group field from the metadata (by @id)
main_df = dataframes[main_rs_id]

# Find a usable numeric field
if len(numeric_fields) > 0:
    numeric_field = numeric_fields[0].id
    print(f"Numeric field used: {numeric_field}")
else:
    print("No numeric field found; cannot continue EDA section.")

threshold = 10  # Example threshold; adjust as needed!
if numeric_field in main_df.columns:
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with '{numeric_field}' > {threshold}.")
    if filtered_df.shape[0] > 0:
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Now find a groupable (categorical) field, as a non-numeric field
        group_field = None
        for field in chosen_record_set.fields:
            if field.data_type == 'Text' and field.id in filtered_df.columns:
                group_field = field.id
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped average of '{numeric_field}' by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print('No records after filtering.')
else:
    print(f"Field '{numeric_field}' not found in columns.")

## 5. Visualization
Visualize the distribution of a numeric field and, if grouping is available, plot by group.

This example uses matplotlib and seaborn for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in main_df.columns and main_df[numeric_field].dtype != 'O':
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    # If group field available and not too many groups, show boxplot
    if 'group_field' in locals() and group_field and main_df[group_field].nunique() < 20:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
        plt.title(f'Boxplot of {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load a Croissant dataset using the `mlcroissant` library
- Explore its schema programmatically referencing all entities by their `@id`
- Extract records from each record set
- Apply basic exploratory data analysis steps
- Visualize results

For more advanced or domain-specific analysis, continue to reference Croissant entities by their `@id` and leverage the extensive metadata for robust, reproducible science.